# Lezione 2 · I dati e il primo classificatore: k-NN

**Intelligenza artificiale · Classe quinta**

Un'idea che tornerà spesso: ogni dato, che sia un fiore, una foto o una parola, si può trasformare in un elenco di numeri, cioè in un **punto** in uno spazio, e dati simili finiscono **vicini**.

Oggi costruiamo questa idea su un dataset che possiamo vedere e toccare: **Iris**, 150 fiori descritti da 4 misure.

> ogni dato è un punto in uno spazio, e due dati **vicini** nello spazio sono **simili**.

Riferimenti sul libro: pp. 218–223 (Pandas ed EDA su Iris).

## 1. Un dato è un elenco di numeri

Per descrivere un fiore, un botanico ha scelto 4 **caratteristiche** (in inglese *features*): lunghezza e larghezza del sepalo e del petalo. Ogni fiore diventa così un elenco di 4 numeri.

Prova: con quali **tre numeri** descriveresti un gatto? Confronta la tua scelta con quella di un compagno. Scegliere le caratteristiche è la prima decisione importante quando si lavora con i dati.

In [ ]:
from sklearn.datasets import load_iris
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

iris = load_iris(as_frame=True)
df = iris.frame
df["specie"] = df["target"].map(dict(enumerate(iris.target_names.tolist())))

print(df.shape)   # (righe, colonne)
df.head()

| colonna | significato |
|---|---|
| `sepal length (cm)` | lunghezza del sepalo |
| `sepal width (cm)` | larghezza del sepalo |
| `petal length (cm)` | lunghezza del petalo |
| `petal width (cm)` | larghezza del petalo |

`specie` dice a quale delle tre specie appartiene il fiore (*setosa*, *versicolor*, *virginica*). Non la useremo per i calcoli: solo per controllare i risultati.

## 2. Esploriamo i dati con Pandas

Prima di usare dei dati bisogna **guardarli**: è l'analisi esplorativa dei dati (EDA). Ripercorriamo i passi del Programma 1 del libro.

`df` è un **DataFrame**, cioè una tabella: una riga per ogni fiore, una colonna per ogni caratteristica. Il metodo `info()` dice quante righe ci sono, di che tipo sono le colonne e se mancano dei valori.

In [ ]:
df.info()

Ogni colonna ha 150 valori non nulli: il dataset è **completo**. Non è scontato: Iris è già stato pulito da qualcuno, più avanti vedremo come si fa con dati che non lo sono.

Con un **filtro** si scelgono le righe che soddisfano una condizione. La condizione produce una colonna di `True`/`False`, e `df[filtro]` tiene solo le righe `True`.

In [ ]:
filtro = df["specie"] == "versicolor"
print(filtro.head())      # i primi 5 fiori sono setose: tutti False
df[filtro].head()         # i primi 5 fiori versicolor

Con `groupby()` si raggruppano le righe per specie e con `mean()` si calcola la media di ogni gruppo: è il fiore "medio" di ogni specie.

In [ ]:
df.drop(columns="target").groupby("specie").mean()

### ✏️ Esercizio 1

Rispondi con Pandas, con una o due righe di codice per domanda:

1. Quanti fiori hanno il petalo più lungo di 5 cm? *(suggerimento: `len()`)*
2. Di che specie sono? *(suggerimento: `value_counts()`)*
3. Qual è il fiore con il sepalo più largo? *(suggerimento: `idxmax()` restituisce l'indice del valore massimo, `df.loc[indice]` la riga)*
4. `df.describe()` calcola minimo, massimo, media… di ogni colonna. Quale misura varia di più tra un fiore e l'altro?

In [ ]:
# 1.

# 2.

# 3.

# 4.

### 🤔 Hai notato?

Le medie dicono già molto: i petali di setosa sono molto più piccoli di quelli delle altre due specie. Ma se arriva un fiore **nuovo**, come facciamo a dire di che specie è? Serve un modo per confrontarlo con i fiori che conosciamo: lo costruiamo nelle prossime sezioni.

## 3. Guardiamo i punti

Con due misure alla volta possiamo disegnare ogni fiore come un punto nel piano.

In [ ]:
colori = {"setosa": "tab:blue", "versicolor": "tab:orange", "virginica": "tab:green"}

for specie, gruppo in df.groupby("specie"):
    plt.scatter(gruppo["sepal length (cm)"], gruppo["sepal width (cm)"],
                label=specie, color=colori[specie])
plt.xlabel("sepal length (cm)")
plt.ylabel("sepal width (cm)")
plt.legend()
plt.show()

### ✏️ Esercizio 2

Cambia le due colonne nel grafico. Quale coppia di misure **separa meglio** le tre specie, cioè forma gruppi di punti più distinti?

In [ ]:
x_col = "..."   # <-- scegli una colonna
y_col = "..."   # <-- scegli un'altra colonna

for specie, gruppo in df.groupby("specie"):
    plt.scatter(gruppo[x_col], gruppo[y_col], label=specie, color=colori[specie])
plt.xlabel(x_col)
plt.ylabel(y_col)
plt.legend()
plt.show()

## 4. Un fiore è un punto in 4D

Nel grafico vediamo 2 misure alla volta, ma ogni fiore ne ha **4**: è un punto in uno spazio a 4 dimensioni. Non possiamo disegnarlo, ma possiamo **calcolare** con esso. Trasformiamo la tabella in una matrice NumPy: ogni riga è un **vettore**.

In [ ]:
X = df.iloc[:, :4].to_numpy()   # 150 righe x 4 colonne
print(X.shape)
print("Fiore 0:  ", X[0])
print("Fiore 100:", X[100])

## 5. La distanza tra due punti

In 2D la distanza si calcola con Pitagora. In 4D (o in 100D) funziona allo stesso modo:

$$d(\mathbf{a}, \mathbf{b}) = \sqrt{(a_1-b_1)^2 + (a_2-b_2)^2 + (a_3-b_3)^2 + (a_4-b_4)^2}$$

### ✏️ Esercizio 3

Completa la funzione `distanza` **senza** usare `np.linalg.norm`. Suggerimento: con NumPy `a - b` sottrae elemento per elemento, e `.sum()` somma tutti gli elementi.

In [ ]:
def distanza(a, b):
    # Il tuo codice qui
    pass

print(distanza(X[0], X[1]))     # due setose
print(distanza(X[0], X[100]))   # setosa e virginica

Controlliamo con la funzione già pronta di NumPy, che d'ora in poi useremo: i risultati devono coincidere.

In [ ]:
print(np.linalg.norm(X[0] - X[1]))
print(np.linalg.norm(X[0] - X[100]))

## 6. Chi sono i vicini?

Calcoliamo in un colpo solo la distanza del fiore 0 da **tutti** gli altri e prendiamo i 5 più vicini.

In [ ]:
fiore = X[0]
dist = np.linalg.norm(X - fiore, axis=1)   # 150 distanze
vicini = np.argsort(dist)[1:6]             # salto il primo: è il fiore stesso
df.iloc[vicini]

Non abbiamo usato la colonna `specie` per trovarli, eppure i vicini sono tutti della stessa specie del fiore 0. **Vicino nello spazio significa simile.**

### ✏️ Esercizio 4

Ripeti la ricerca per il fiore **70** e per il fiore **120**. I 5 vicini sono sempre della stessa specie? Dove ci sono "errori", e perché proprio lì? (Riguarda il grafico dell'esercizio 1.)

In [ ]:
for indice in [70, 120]:
    # Il tuo codice qui: trova i 5 vicini e stampa la loro specie
    pass

## 7. Il primo classificatore: k-NN

Se i vicini di un fiore sono quasi tutti della stessa specie, possiamo usarli per **indovinare** la specie di un fiore nuovo: cerchiamo i suoi *k* vicini più prossimi e scegliamo la specie **più frequente** tra loro (voto di maggioranza).

Questo algoritmo si chiama **k-NN** (*k-nearest neighbors*, "i k vicini più prossimi"). È un algoritmo di apprendimento **supervisionato**, perché usa le etichette (le specie) dei fiori già noti.

Per contare i voti usiamo `Counter` del modulo `collections`:

In [ ]:
from collections import Counter

voti = Counter(["setosa", "virginica", "setosa"])
print(voti.most_common(1))   # [(elemento più frequente, quante volte)]

### ✏️ Esercizio 5

Completa la funzione `knn(nuovo_fiore, k)`: calcola le distanze di `nuovo_fiore` da tutti i fiori di `X`, prende i `k` più vicini e restituisce la specie più frequente tra loro.

Poi prova con due fiori misurati "sul campo" e con valori diversi di `k`.

In [ ]:
def knn(nuovo_fiore, k):
    # Il tuo codice qui
    pass

print(knn(np.array([5.0, 3.4, 1.5, 0.2]), k=5))   # atteso: setosa
print(knn(np.array([6.5, 3.0, 5.5, 2.0]), k=5))   # atteso: virginica
print(knn(np.array([6.0, 2.8, 4.9, 1.6]), k=1), knn(np.array([6.0, 2.8, 4.9, 1.6]), k=15))   # un caso al confine

### 🤔 Quanto vale k?

Con `k = 1` decide un solo vicino, che potrebbe essere un'eccezione; con `k` molto grande votano anche fiori lontani. La scelta di `k` non la impara l'algoritmo: la decide chi lo usa. È un **iperparametro**, come quelli che incontreremo addestrando le reti neurali.

## 8. Il k-NN funziona davvero? Training set e test set

Finora abbiamo provato il k-NN su pochi fiori inventati. Per sapere **quanto** funziona dobbiamo misurarlo su fiori di cui conosciamo la specie, ma che il classificatore **non ha mai visto**.

Dividiamo quindi il dataset in due parti:

- il **training set** (circa 80% dei fiori): sono i fiori "noti", tra cui il k-NN cerca i vicini;
- il **test set** (circa 20%): fiori che teniamo da parte e usiamo solo per verificare le risposte.

L'**accuratezza** è la percentuale di fiori del test set classificati correttamente.

In [ ]:
rng = np.random.default_rng(42)          # generatore casuale con seme fisso: risultati ripetibili
ordine = rng.permutation(len(X))          # i numeri da 0 a 149 mescolati
train, test = ordine[:120], ordine[120:]

X_train, y_train = X[train], df["specie"].to_numpy()[train]
X_test,  y_test  = X[test],  df["specie"].to_numpy()[test]
print(len(X_train), "fiori nel training set,", len(X_test), "nel test set")

### ✏️ Esercizio 6

Completa la funzione `knn_train(nuovo_fiore, k)`, uguale a `knn` ma che cerca i vicini **solo nel training set**. Poi completa `accuratezza(k)`, che classifica tutti i fiori del test set e restituisce la frazione di risposte giuste.

In [ ]:
def knn_train(nuovo_fiore, k):
    # Il tuo codice qui
    pass

def accuratezza(k):
    # Il tuo codice qui
    pass

print(accuratezza(5))

### ✏️ Esercizio 7

Calcola l'accuratezza per `k` da 1 a 30 e disegna il grafico. Quale `k` sceglieresti? Prova poi a cambiare il seme del generatore casuale (42): il grafico cambia?

In [ ]:
# Il tuo codice qui

## 🔗 E se i dati fossero parole?

- Ogni **parola** può diventare un punto in uno spazio con centinaia di dimensioni, invece che 4.
- Cercare le parole **simili** a una parola data significa cercare i suoi vicini, come avete fatto nella sezione 6.
- La **ricerca semantica**, il cuore dei sistemi che rispondono usando dei documenti, è la stessa cosa con le frasi: trova i documenti più vicini alla domanda.

La differenza fondamentale: le 4 caratteristiche di Iris le ha scelte un botanico; le 100 coordinate delle parole le ha **imparate un algoritmo** leggendo milioni di testi. Nessuno sa dire che cosa significhi, da sola, ciascuna di esse.